# Course 20 lab — Multi-repository SDD

This notebook imports the same trusted harness and control plane used by the tests. It creates five real temporary Git repositories and keeps agent claims separate from independently derived evidence.

## 1. Establish the exact integration subject

A system result is meaningful only when it names the parent change, contract revision, and immutable commit for every component.

In [ ]:
import importlib.util, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
spec = importlib.util.spec_from_file_location('course20_notebook_lab', Path('lab.py'))
course20_lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = course20_lab
spec.loader.exec_module(course20_lab)
build_candidate = course20_lab.build_candidate
assess_candidate = course20_lab.assess_candidate
build_report = course20_lab.build_report
candidate = build_candidate()
decision, context = assess_candidate(candidate)
assert decision.state == 'READY_FOR_INTEGRATION_REVIEW'
candidate['integration_manifest']

## 2. Inspect actual Git revisions and diffs

The agent does not declare its own commit identity or changed-path evidence. The harness initializes, commits, and compares each repository.

In [ ]:
revisions = {name: {'baseline': item.baseline_commit, 'candidate': item.candidate_commit, 'paths': item.changed_paths} for name, item in context.repositories.items()}
assert len(revisions) == 5
assert all(len(item['candidate']) == 40 for item in revisions.values())
revisions

## 3. Derive mixed-version compatibility

The new consumer accepts revisions 1 and 2; the old consumer accepts only revision 1. That makes deployment order a property of the matrix, not a preference.

In [ ]:
matrix = dict(context.compatibility_matrix)
assert matrix['ai-v1/review-v2'] is True
assert matrix['ai-v2/review-v1'] is False
matrix

## 4. Compare local success with system evidence

A cross-tenant defect does not need to break repository-local tests. The governed evaluator must still block the integrated change.

In [ ]:
unsafe = build_candidate(['cross_tenant_leak'])
unsafe_decision, unsafe_context = assess_candidate(unsafe)
assert all(unsafe['local_ci'].values())
assert unsafe_decision.state == 'BLOCKED'
[(finding.code, finding.message) for finding in unsafe_decision.findings]

## 5. Protect system authority

AI Service may create proposals. It cannot become a second owner of underwriting state.

In [ ]:
dual_write = build_candidate(['dual_write_authority'])
dual_decision, _ = assess_candidate(dual_write)
assert 'SYSTEM_INTEGRATION_FAILED' in {finding.code for finding in dual_decision.findings}
dual_decision.state

## 6. Reject self-reported integration success

The candidate can submit results, but trusted execution recomputes them and reports mismatches.

In [ ]:
false_report = build_candidate(['cross_tenant_leak', 'false_integration_report'])
false_decision, _ = assess_candidate(false_report)
false_codes = {finding.code for finding in false_decision.findings}
assert {'INTEGRATION_REPORT_MISMATCH', 'SYSTEM_INTEGRATION_FAILED'} <= false_codes
sorted(false_codes)

## 7. Verify rollback compatibility

If Review v2 rejects AI v1, restoring the old producer cannot recover the system. The evaluator derives this from executed compatibility rather than a checkbox.

In [ ]:
bad_rollback = build_candidate(['new_consumer_rejects_v1'])
rollback_decision, _ = assess_candidate(bad_rollback)
assert 'ROLLBACK_COMPATIBILITY_UNVERIFIED' in {finding.code for finding in rollback_decision.findings}
rollback_decision.state

## 8. Run the complete labelled evaluation

The counts below establish reproducibility for this declared population. They do not measure general agent accuracy or authorize production release.

In [ ]:
report = build_report()
assert report['conformance']['passed'] == report['conformance']['total'] == 16
assert report['evaluation']['passed'] == report['evaluation']['total'] == 40
assert report['integration_scenarios']['passed'] == report['integration_scenarios']['total'] == 11
{key: report[key] for key in ('course_validation', 'candidate_assessment', 'comparison', 'coordination')}

## Production upgrades and exercises

Production needs authenticated workload and owner identities, signed provenance, isolated candidate execution, durable orchestration, real contract tooling, deployment receipts, runtime version telemetry, consumer inventory, and accountable approval.

Exercises: add a hidden batch consumer; model a revision-3 contract proposal; inject a duplicate after restart; and design evidence that would justify—not merely assert—revision-1 retirement.